# Notebook 4: Motif Extended Characterization

This notebook extends `nb03` with the broader motif-characterization experiments:

1. `motif_cooccurrence_graph`: do motifs form overlap-based families?
2. `motif_phase_match`: which motifs persist, split, merge, or appear uniquely after Phase C?
3. `motif_topology`: are motifs spatial, depth-like, or fragmented?
4. `motif_transfer_stability`: do motifs rediscovered on overlapping subsamples remain similar?

This notebook requires cached `motif_families.json` artifacts from `nb03` and will fail loudly if they are missing.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.evaluation.motif_validation import (
    EXTENDED_MOTIF_EXPERIMENT_IDS,
    run_motif_cooccurrence_experiment,
    run_motif_phase_match_experiment,
    run_motif_topology_experiment,
    run_motif_transfer_stability_experiment,
)
from flow_circuits.training import load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all four extended motif experiments or any subset.
This notebook depends on cached motif-family artifacts from `nb03` and will not rediscover them silently.


In [ ]:
from pathlib import Path

RUN_MODE = "debug"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
NB03_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_recurring_motif_core_validation" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb04_motif_extended_characterization" / CONFIG_NAME

PHASE_B_CHECKPOINT = EXPERIMENT_ROOT / "phase_b.pt"
PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import time

import torch

try:
    import pandas as pd
except ImportError:
    pd = None

MODEL_ORDER = [("phase_b", "Phase B"), ("phase_c", "Phase C")]
EXPERIMENT_LABELS = {
    "motif_cooccurrence_graph": "Motif Co-occurrence Graph",
    "motif_phase_match": "Motif Phase Match",
    "motif_topology": "Motif Topology",
    "motif_transfer_stability": "Motif Transfer Stability",
}

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(EXTENDED_MOTIF_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(EXTENDED_MOTIF_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {EXTENDED_MOTIF_EXPERIMENT_IDS}")

if not PHASE_B_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_B_CHECKPOINT}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

REQUIRED_NB03_ARTIFACTS = {
    "phase_b": NB03_OUTPUT_DIR / "phase_b" / "motif_families.json",
    "phase_c": NB03_OUTPUT_DIR / "phase_c" / "motif_families.json",
}
for tag, artifact_path in REQUIRED_NB03_ARTIFACTS.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing required motif artifact from nb03: {artifact_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "motif_cooccurrence_graph": {"overlap_threshold": 0.25},
        "motif_phase_match": {},
        "motif_topology": {},
        "motif_transfer_stability": {"max_images": 1000, "nodes_per_layer": 2, "bootstrap_iterations": 2},
    },
    "full": {
        "motif_cooccurrence_graph": {"overlap_threshold": 0.25},
        "motif_phase_match": {},
        "motif_topology": {},
        "motif_transfer_stability": {"max_images": 5000, "nodes_per_layer": 4, "bootstrap_iterations": 5},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"RUN_MODE must be one of {sorted(SETTINGS_BY_MODE)}")


def _format_seconds(seconds):
    if seconds is None:
        return "?"
    seconds = int(max(0, round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    raw_label = event["checkpoint_tag"]
    label = {"phase_b": "Phase B", "phase_c": "Phase C"}.get(raw_label, raw_label)
    stage = event["stage"].replace("_", " ")
    total = event.get("total")
    counter = stage if total is None else f"{stage} {event['completed']}/{total}"
    elapsed = _format_seconds(event.get("elapsed_seconds"))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if eta is None:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | {message}", flush=True)
    else:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | ETA {_format_seconds(eta)} | {message}", flush=True)


def _cache_path(tag, experiment_id):
    return OUTPUT_DIR / tag / f"{experiment_id}.json"


def _load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _write_summary_table(rows, *, title):
    print(title)
    if not rows:
        print("  No rows to display.")
        return
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Run mode      : {RUN_MODE}")
print(f"Config        : {CONFIG_NAME}")
print(f"Experiments   : {SELECTED_EXPERIMENTS}")
print(f"Phase B ckpt  : {PHASE_B_CHECKPOINT}")
print(f"Phase C ckpt  : {PHASE_C_CHECKPOINT}")
print(f"nb03 root     : {NB03_OUTPUT_DIR}")
print(f"Output dir    : {OUTPUT_DIR}")

components_by_tag = {
    "phase_b": load_components_from_checkpoint(PHASE_B_CHECKPOINT, device=device),
    "phase_c": load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device),
}
base_config = components_by_tag["phase_b"].config
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=int(base_config["data"]["batch_size"]),
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

RESULTS = {}
MOTIF_ARTIFACTS = {tag: _load_json(path) for tag, path in REQUIRED_NB03_ARTIFACTS.items()}


def _run_or_load(tag, experiment_id, run_fn):
    cache_path = _cache_path(tag, experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id} for {tag}: {cache_path}")
        return _load_json(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Running {experiment_id} for {tag}; cache -> {cache_path}")
    return run_fn(cache_path)


## Experiment 1: Motif Co-occurrence Graph

Build an overlap graph over motif member sets to see whether motifs form broader families or communities.


In [ ]:
if _should_run("motif_cooccurrence_graph"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_cooccurrence_graph"]
    cooccurrence_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = MOTIF_ARTIFACTS[tag]
        cooccurrence_results[tag] = _run_or_load(
            tag,
            "motif_cooccurrence_graph",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_cooccurrence_experiment(
                motif_artifact,
                checkpoint_tag=tag,
                overlap_threshold=settings["overlap_threshold"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_cooccurrence_graph"] = cooccurrence_results
else:
    print("Skipping motif_cooccurrence_graph.")


In [ ]:
if "motif_cooccurrence_graph" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_cooccurrence_graph"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "n_motifs": summary["n_motifs"],
            "n_edges": summary["n_edges"],
            "graph_density": summary["graph_density"],
            "largest_component_size": summary["largest_component_size"],
        })
    _write_summary_table(rows, title="Motif co-occurrence summary")


## Experiment 2: Motif Phase Match

Match Phase B motifs to Phase C motifs by overlap and structural similarity to see which motifs persist, split, merge, or appear uniquely after alignment.


In [ ]:
if _should_run("motif_phase_match"):
    RESULTS["motif_phase_match"] = _run_or_load(
        "comparisons",
        "motif_phase_match",
        lambda cache_path: run_motif_phase_match_experiment(
            MOTIF_ARTIFACTS["phase_b"],
            MOTIF_ARTIFACTS["phase_c"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("Skipping motif_phase_match.")


In [ ]:
if "motif_phase_match" in RESULTS:
    summary = RESULTS["motif_phase_match"]["summary"]
    _write_summary_table([
        {
            "matched_motif_count": summary["matched_motif_count"],
            "mean_match_quality": summary["mean_match_quality"],
            "phase_c_only_count": len(RESULTS["motif_phase_match"]["phase_c_only_motifs"]),
            "phase_b_unmatched_count": len(RESULTS["motif_phase_match"]["phase_b_unmatched_motifs"]),
        }
    ], title="Motif phase-match summary")


## Experiment 3: Motif Topology

Characterize motifs as spatial, depth-like, or fragmented based on their active-node structure.


In [ ]:
if _should_run("motif_topology"):
    topology_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = MOTIF_ARTIFACTS[tag]
        topology_results[tag] = _run_or_load(
            tag,
            "motif_topology",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_topology_experiment(
                motif_artifact,
                checkpoint_tag=tag,
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_topology"] = topology_results
else:
    print("Skipping motif_topology.")


In [ ]:
if "motif_topology" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_topology"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "mean_same_layer_adjacent_edges": summary["mean_same_layer_adjacent_edges"],
            "mean_same_cell_depth_edges": summary["mean_same_cell_depth_edges"],
            "mean_largest_connected_component": summary["mean_largest_connected_component"],
            "topology_counts": summary["topology_counts"],
        })
    _write_summary_table(rows, title="Motif topology summary")


## Experiment 4: Motif Transfer Stability

Rediscover motifs on overlapping discovery subsamples and check whether the resulting motifs remain similar.


In [ ]:
if _should_run("motif_transfer_stability"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_transfer_stability"]
    transfer_results = {}
    for tag, _label in MODEL_ORDER:
        transfer_results[tag] = _run_or_load(
            tag,
            "motif_transfer_stability",
            lambda cache_path, tag=tag: run_motif_transfer_stability_experiment(
                components_by_tag[tag],
                loaders["discovery"],
                device=device,
                checkpoint_tag=tag,
                max_images=settings["max_images"],
                nodes_per_layer=settings["nodes_per_layer"],
                bootstrap_iterations=settings["bootstrap_iterations"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_transfer_stability"] = transfer_results
else:
    print("Skipping motif_transfer_stability.")


In [ ]:
if "motif_transfer_stability" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_transfer_stability"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "matched_motif_rate": summary["matched_motif_rate"],
            "mean_image_set_stability": summary["mean_image_set_stability"],
            "mean_support_node_stability": summary["mean_support_node_stability"],
            "mean_exemplar_overlap": summary["mean_exemplar_overlap"],
        })
    _write_summary_table(rows, title="Motif transfer-stability summary")


## Final Comparison

This last cell collects the main summary metric from each selected extended experiment for quick comparison.


In [ ]:
comparison_rows = []
if "motif_cooccurrence_graph" in RESULTS:
    comparison_rows.append({
        "experiment": EXPERIMENT_LABELS["motif_cooccurrence_graph"],
        "phase_b": RESULTS["motif_cooccurrence_graph"]["phase_b"]["summary"]["graph_density"],
        "phase_c": RESULTS["motif_cooccurrence_graph"]["phase_c"]["summary"]["graph_density"],
        "metric": "graph_density",
    })
if "motif_phase_match" in RESULTS:
    comparison_rows.append({
        "experiment": EXPERIMENT_LABELS["motif_phase_match"],
        "phase_b": RESULTS["motif_phase_match"]["summary"]["matched_motif_count"],
        "phase_c": len(RESULTS["motif_phase_match"]["phase_c_only_motifs"]),
        "metric": "matched_vs_phase_c_only",
    })
if "motif_topology" in RESULTS:
    comparison_rows.append({
        "experiment": EXPERIMENT_LABELS["motif_topology"],
        "phase_b": RESULTS["motif_topology"]["phase_b"]["summary"]["mean_largest_connected_component"],
        "phase_c": RESULTS["motif_topology"]["phase_c"]["summary"]["mean_largest_connected_component"],
        "metric": "mean_largest_connected_component",
    })
if "motif_transfer_stability" in RESULTS:
    comparison_rows.append({
        "experiment": EXPERIMENT_LABELS["motif_transfer_stability"],
        "phase_b": RESULTS["motif_transfer_stability"]["phase_b"]["summary"]["matched_motif_rate"],
        "phase_c": RESULTS["motif_transfer_stability"]["phase_c"]["summary"]["matched_motif_rate"],
        "metric": "matched_motif_rate",
    })
_write_summary_table(comparison_rows, title="Motif extended characterization summary")
(COMPARISON_DIR / "summary.json").write_text(json.dumps(comparison_rows, indent=2), encoding="utf-8")
